[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/02-data-harmonization/01-character_mapping.ipynb)

# Character Mapping

In real-world data, names, addresses, and other text fields often show up in more than one valid spelling. Accents get dropped, umlauts get typed out differently, and casing is inconsistent from one system to the next. `CharacterMapping` lets you standardize text before it's indexed and before a query is compared against it, so both sides land on the same representation.

In this notebook you will:

1. Build an index without any character mapping, and see how the default matcher handles a German name
2. Inspect what `CharacterMapping.create_german_standard()` actually does to text
3. Attach it to your index and see how matching changes
4. Know when to reach for a `CharacterMapping` in your own data


## 1. A dataset with some German names

Umlauts (`ä`, `ö`, `ü`) and the sharp-s (`ß`) are common in German text, and they get typed and stored inconsistently,  sometimes as the accented character, sometimes spelled out (`ü` as `ue`), sometimes just dropped entirely.

In [1]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.DataFrame({
    "full_name": ["Jürgen Müller", "Käthe Björnsson", "Grüning GmbH", "Weiß & Co."],
    "city": ["München", "Köln", "Nürnberg", "Düsseldorf"]
})

df

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,full_name,city
0,Jürgen Müller,München
1,Käthe Björnsson,Köln
2,Grüning GmbH,Nürnberg
3,Weiß & Co.,Düsseldorf


Let's build an index with no character mapping at all, and query it two different ways for the same person: once with the umlaut simply dropped, and once with it spelled out the way German speakers do when they don't have an umlaut key,  `ü` written as `ue`.

In [3]:
baseline_index = TableIndexer.create_index(df,
                                           index_columns=["full_name", "city"],
                                           tmp_dir="tmp_index")

baseline_results = baseline_index.match(
    full_name=["Jurgen Muller", "Juergen Mueller"],
    city=["Munchen", "Muenchen"],
    include_field_scores=True,
    include_queries=True
)

baseline_results

,query_row,full_name_query,city_query,index_row,full_name_candidate,city_candidate,overall_score,full_name_score,city_score
0,0,Jurgen Muller,Munchen,0,Jürgen Müller,München,100,100,100
1,1,Juergen Mueller,Muenchen,0,Jürgen Müller,München,71,69,74


Take a look at `full_name_score` for both rows. You might expect `"Juergen Mueller"`,  the properly spelled-out version,  to score at least as well as `"Jurgen Muller"`. Depending on the result you see, it may not: the matcher is comparing raw characters, and `"Mueller"` has an extra letter compared to `"Müller"` that `"Muller"` doesn't. Character-for-character, dropping the umlaut can look like a smaller change than spelling it out correctly,  even though spelling it out is the more meaningful, standard convention.

That's the gap `CharacterMapping` is built to close.

## 2. What `create_german_standard()` does

`CharacterMapping` ships with ready-made factory methods for common cases. `create_german_standard()` handles German text specifically,  expanding umlauts, removing other accents, and standardizing case. You can test it directly on plain strings before attaching it to anything.

In [4]:
from mbox.mapping import CharacterMapping

german_mapping = CharacterMapping.create_german_standard()

print("Müller  ->", german_mapping.apply("Müller"))
print("Mueller ->", german_mapping.apply("Mueller"))
print("Muller  ->", german_mapping.apply("Muller"))

Müller  -> MUELLER
Mueller -> MUELLER
Muller  -> MULLER


Notice that `"Müller"` and `"Mueller"` normalize to the exact same string. `"Muller"` doesn't,  it's missing a letter that the other two share once normalized. This is `expand_umlauts` at work: it rewrites `ü` as `UE`, rather than just stripping the accent down to a plain `U`. The standard, spelled-out German convention and the original accented spelling become identical after normalization,  while a casually dropped umlaut stays distinguishable from both.

You can also normalize an entire column at once with `apply_series()`, which is handy for previewing what an index will store before you build it:

In [5]:
df["full_name_normalized"] = german_mapping.apply_series(df["full_name"])
df[["full_name", "full_name_normalized"]]

,full_name,full_name_normalized
0,Jürgen Müller,JUERGEN MUELLER
1,Käthe Björnsson,KAETHE BJOERNSSON
2,Grüning GmbH,GRUENING GMBH
3,Weiß & Co.,WEISS CO


## 3. Attach the mapping to your index

In practice, you don't normalize columns by hand. You attach a `CharacterMapping` to a field during index creation with `character_mappings`, and M|BOX applies it automatically,  to the indexed data *and* to every incoming query. Let's rebuild the index this way, and run the exact same two queries from Step 1.

In [7]:
harmonized_index = TableIndexer.create_index(
    df=df,
    index_columns=["full_name", "city"],
    character_mappings={
        "full_name": german_mapping,
        "city": german_mapping
    },
    tmp_dir="tmp_index"
)

harmonized_results = harmonized_index.match(
    full_name=["Jurgen Muller", "Juergen Mueller"],
    city=["Munchen", "Muenchen"],
    include_field_scores=True,
    include_queries=True
)

harmonized_results

,query_row,full_name_query,city_query,index_row,full_name_candidate,city_candidate,full_name_normalized_candidate,overall_score,full_name_score,city_score
0,0,Jurgen Muller,Munchen,0,Jürgen Müller,München,JUERGEN MUELLER,70,66,74
1,1,Juergen Mueller,Muenchen,0,Jürgen Müller,München,JUERGEN MUELLER,100,100,100


Compare the two `full_name_score` values here against what you saw in Step 1. The properly spelled-out query, `"Juergen Mueller"`, should now score at least as well as,  and likely better than,  the version with the umlaut simply dropped. Harmonizing at index time means the engine recognizes `ü`, `ue`, and the standard transliteration as pointing at the same underlying text, instead of leaving that up to raw character overlap.

## 4. When to reach for `CharacterMapping`

Use a `CharacterMapping` whenever a field can legitimately be written more than one way, and you want the engine to treat those variations as equivalent rather than penalizing them:

- **Accented and non-accented text**,  names, addresses, and companies from regions where diacritics are common but frequently dropped in typing or data entry
- **Casing inconsistency**,  one system stores `"BERLIN"`, another stores `"Berlin"`
- **Currency and special characters**,  normalizing symbols so `"$100"` and `"USD 100"` don't silently diverge

M|BOX ships factory methods for the most common cases (`create_english_standard()`, `create_german_standard()`, `create_strict_alphanumeric()`), and you can also build a fully custom `CharacterMapping` for rules specific to your own data,  covered next.

## Next steps

- **`alias_sets_nicknames_and_synonyms.ipynb`**,  handle *meaning* equivalence (Bob/Robert, IBM/International Business Machines), not just spelling
- **`building_a_custom_character_mapping.ipynb`**,  write your own normalization rules for cases the built-in standards don't cover
- **`combining_mappings_and_aliases_in_one_index.ipynb`**,  use `CharacterMapping` and `AliasSet` together on the same field

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*